# Feature Engineering v4 — F1 Strategic Decision Engine

## Arquitectura de dos capas (separadas)

| Capa | Dataset | Unidad de análisis | Target de features | Uso |
|---|---|---|---|---|
| **Telemetría** | `df_master` (3331 filas) | 1 fila = 1 vuelta de 1 piloto | ~30 variables | PCA → Driver Performance States |
| **Táctica** | `df_events` (643 filas) | 1 fila = 1 evento táctico | ~25 variables | Clustering → Strategy Archetypes |

> **Regla de oro:** NO mezclar capas. El puente entre ellas son los PC scores
> agregados de telemetría que se añaden como features al clustering táctico.

In [30]:
import polars as pl
import numpy as np
import polars.selectors as cs
from pathlib import Path

print(f'Polars: {pl.__version__}')
pl.Config.set_tbl_cols(20)

BASE_DIR = Path('../../data')
CARRERAS = ['australia', 'china', 'japan', 'united_states']

Polars: 1.38.1


---
## 1. Carga de datos

In [31]:
# ── Telemetría por vuelta ──────────────────────────────────────────────────
master_dfs = []
for c in CARRERAS:
    p = f'../../data/processed/{c}/{c}_2026_masterv2.parquet'
    master_dfs.append(
            pl.read_parquet(p)
            .with_columns(cs.integer().cast(pl.Float64))
            .with_columns(pl.lit(c).alias('race_name'))
    )

df_master = pl.concat(master_dfs, how='diagonal')

# ── Eventos tácticos ───────────────────────────────────────────────────────
event_dfs = []
for c in CARRERAS:
    p = f'../../data/events/{c}_2026_events.parquet'
    event_dfs.append(
            pl.read_parquet(p)
            .with_columns(cs.integer().cast(pl.Float64))
            .with_columns(pl.lit(c).alias('race_name'))
        )

df_events = pl.concat(event_dfs, how='diagonal')

print(f'df_master : {df_master.shape}')
print(f'df_events : {df_events.shape}')
print(f'Columnas master : {df_master.columns}')
print(f'Columnas events : {df_events.columns}')

df_master : (3331, 29)
df_events : (643, 8)
Columnas master : ['meeting_key', 'session_key', 'driver_number', 'lap_number', 'date_start', 'duration_sector_1', 'duration_sector_2', 'duration_sector_3', 'i1_speed', 'i2_speed', 'is_pit_out_lap', 'lap_duration', 'segments_sector_1', 'segments_sector_2', 'segments_sector_3', 'st_speed', 'position', 'compound', 'stint_number', 'tyre_age', 'pit_duration', 'is_pit_lap', 'throttle_mean_lap', 'throttle_std_lap', 'brake_max_lap', 'rpm_max_lap', 'n_gear_max_lap', 'drs_max_lap', 'race_name']
Columnas events : ['race_id', 'lap_number', 'event_type', 'initiator_driver', 'target_driver', 'initiator_compound', 'initiator_pos_change', 'race_name']


In [32]:
df_master.columns

['meeting_key',
 'session_key',
 'driver_number',
 'lap_number',
 'date_start',
 'duration_sector_1',
 'duration_sector_2',
 'duration_sector_3',
 'i1_speed',
 'i2_speed',
 'is_pit_out_lap',
 'lap_duration',
 'segments_sector_1',
 'segments_sector_2',
 'segments_sector_3',
 'st_speed',
 'position',
 'compound',
 'stint_number',
 'tyre_age',
 'pit_duration',
 'is_pit_lap',
 'throttle_mean_lap',
 'throttle_std_lap',
 'brake_max_lap',
 'rpm_max_lap',
 'n_gear_max_lap',
 'drs_max_lap',
 'race_name']

In [33]:
import requests

# ── Obtener team_name desde OpenF1 /drivers ───────────────────────────────
SESSION_KEYS = df_master.select('session_key').unique().to_series().to_list()

drivers_list = []
for sk in SESSION_KEYS:
    url = f'https://api.openf1.org/v1/drivers?session_key={sk}'
    r = requests.get(url)
    if r.status_code == 200:
        drivers_list.extend(r.json())

df_drivers = (
    pl.DataFrame(drivers_list)
      .select(['session_key', 'driver_number', 'team_name', 'name_acronym'])
      .unique(subset=['session_key', 'driver_number'])
      .with_columns([
          pl.col('session_key').cast(pl.Float64),    # ← match con df_master
          pl.col('driver_number').cast(pl.Float64),  # ← por si acaso
      ])
)

# Join al df_master
df_master = df_master.join(df_drivers, on=['session_key', 'driver_number'], how='left')

print(df_master.select(['driver_number', 'team_name', 'name_acronym']).unique().sort('team_name'))

shape: (22, 3)
┌───────────────┬─────────────────┬──────────────┐
│ driver_number ┆ team_name       ┆ name_acronym │
│ ---           ┆ ---             ┆ ---          │
│ f64           ┆ str             ┆ str          │
╞═══════════════╪═════════════════╪══════════════╡
│ 10.0          ┆ Alpine          ┆ GAS          │
│ 43.0          ┆ Alpine          ┆ COL          │
│ 18.0          ┆ Aston Martin    ┆ STR          │
│ 14.0          ┆ Aston Martin    ┆ ALO          │
│ 27.0          ┆ Audi            ┆ HUL          │
│ …             ┆ …               ┆ …            │
│ 41.0          ┆ Racing Bulls    ┆ LIN          │
│ 6.0           ┆ Red Bull Racing ┆ HAD          │
│ 3.0           ┆ Red Bull Racing ┆ VER          │
│ 55.0          ┆ Williams        ┆ SAI          │
│ 23.0          ┆ Williams        ┆ ALB          │
└───────────────┴─────────────────┴──────────────┘


In [34]:
df_master.head()

meeting_key,session_key,driver_number,lap_number,date_start,duration_sector_1,duration_sector_2,duration_sector_3,i1_speed,i2_speed,…,is_pit_lap,throttle_mean_lap,throttle_std_lap,brake_max_lap,rpm_max_lap,n_gear_max_lap,drs_max_lap,race_name,team_name,name_acronym
f64,f64,f64,f64,datetime[μs],f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,str,str,str,str
1279.0,11234.0,6.0,2.0,2026-03-08 04:06:28.913,30.094,18.602,36.86,238.0,276.0,…,0.0,66.350769,40.947648,104.0,11886.0,8.0,null,"""australia""","""Red Bull Racing""","""HAD"""
1279.0,11234.0,41.0,2.0,2026-03-08 04:06:30.170,30.171,17.966,37.36,255.0,304.0,…,0.0,null,null,null,null,null,null,"""australia""","""Racing Bulls""","""LIN"""
1279.0,11234.0,1.0,2.0,2026-03-08 04:06:30.675,30.541,18.252,38.07,245.0,311.0,…,0.0,60.924699,44.222423,104.0,12704.0,8.0,null,"""australia""","""McLaren""","""NOR"""
1279.0,11234.0,12.0,2.0,2026-03-08 04:06:30.967,30.586,18.042,36.835,256.0,304.0,…,0.0,null,null,null,null,null,null,"""australia""","""Mercedes""","""ANT"""
1279.0,11234.0,31.0,2.0,2026-03-08 04:06:31.392,30.865,17.978,37.724,null,308.0,…,0.0,null,null,null,null,null,null,"""australia""","""Haas F1 Team""","""OCO"""


---
## 2. CAPA A — Features de telemetría (para PCA)

**Unidad:** 1 fila = 1 vuelta de 1 piloto  
**Objetivo:** ~30 variables que describan el *estado de rendimiento* del piloto en esa vuelta  
**No incluir:** datos de otros pilotos, intervalos, ni contexto táctico

### Variables objetivo (30 total)

| Grupo | Variables | N |
|---|---|---|
| Ritmo puro | `lap_duration`, `sector_1`, `sector_2`, `sector_3` | 4 |
| Velocidad | `speed_trap`, `speed_i1`, `speed_i2` | 3 |
| Estilo de conducción | `throttle_mean`, `throttle_pct_full`, `brake_pct_active`, `coasting_pct` | 4 |
| Motor | `rpm_max`, `rpm_mean`, `gear_max` | 3 |
| Neumático | `tyre_age`, `compound_ord`, `deg_rate_stint` | 3 |
| Posición en carrera | `position`, `gap_to_leader` | 2 |
| Contexto stint | `stint_number`, `is_pit_out_lap`, `is_pit_lap` | 3 |
| Eficiencias derivadas | `speed_per_rpm`, `sector1_pct`, `sector3_pct`, `sector_balance`, `throttle_brake_ratio`, `lap_vs_best_stint` | 6 |
| Identificadores (no entran al PCA) | `race_name`, `driver_number`, `lap_number` | — |

In [35]:
# ── Helpers ────────────────────────────────────────────────────────────────

def safe_div(a, b):
    """División segura en expresiones Polars."""
    return pl.when(b != 0).then(a / b).otherwise(None)


# ✅ Después — todos float explícitos
COMPOUND_ORD = {'SOFT': 1.0, 'MEDIUM': 2.0, 'HARD': 3.0, 'INTERMEDIATE': 1.5, 'WET': 0.5}


def build_telemetry_features(df: pl.DataFrame) -> pl.DataFrame:
    """
    Construye ~30 features de telemetría por vuelta.
    Input : df_master con columnas base de OpenF1
    Output: df con columnas de features + identificadores
    """
    df = df.with_columns([
        # ── Compound ordinal ─────────────────────────────────────────────
        pl.col('compound').replace_strict(
      ['SOFT', 'MEDIUM', 'HARD', 'INTERMEDIATE', 'WET'],
      [1.0,    2.0,      3.0,    1.5,             0.5],
      default=None,
      return_dtype=pl.Float64
  )
  .alias('compound_ord')
    ])

    # ── Ritmo sectorial (% del tiempo total por sector) ──────────────────
    lap_col = pl.col('lap_duration')
    df = df.with_columns([
        safe_div(pl.col('duration_sector_1'), lap_col).alias('sector1_pct'),
        safe_div(pl.col('duration_sector_2'), lap_col).alias('sector2_pct'),
        safe_div(pl.col('duration_sector_3'), lap_col).alias('sector3_pct'),
        safe_div(pl.col('duration_sector_1'), pl.col('duration_sector_3')).alias('sector_balance'),
    ])

    # ── Estilo de conducción ─────────────────────────────────────────────
    df = df.with_columns([
        # Porcentaje de la vuelta con throttle > 95% (estimado desde throttle_mean_lap)
        pl.when(pl.col('throttle_mean_lap') > 95).then(1.0).otherwise(
            safe_div(pl.col('throttle_mean_lap'), pl.lit(100.0))
        ).alias('throttle_pct_full'),

        # Ratio throttle / brake (mayor = más agresivo en aceleración)
        safe_div(pl.col('throttle_mean_lap'), pl.col('brake_max_lap')).alias('throttle_brake_ratio'),

        # Proxy de coasting: si brake y throttle son bajos simultáneamente
        pl.when(
            (pl.col('throttle_mean_lap') < 10) & (pl.col('brake_max_lap') < 10)
        ).then(1.0).otherwise(0.0).alias('coasting_pct'),
    ])

    # ── Motor: eficiencia velocidad/RPM ──────────────────────────────────
    df = df.with_columns([
        safe_div(pl.col('st_speed'), pl.col('rpm_max_lap')).alias('speed_per_rpm'),
    ])

    # ── Neumático: tasa de degradación dentro del stint ──────────────────
    # Calculamos el mejor lap_duration del stint hasta esa vuelta (rolling min por stint)
    df = df.sort(['race_name', 'driver_number', 'stint_number', 'lap_number'])
    df = df.with_columns([
        pl.col('lap_duration')
          .min()
          .over(['race_name', 'driver_number', 'stint_number'])
          .alias('best_lap_stint'),
    ])
    df = df.with_columns([
        # Cuánto más lento que la mejor vuelta del stint (degradación acumulada)
        safe_div(
            pl.col('lap_duration') - pl.col('best_lap_stint'),
            pl.col('best_lap_stint')
        ).alias('lap_vs_best_stint'),
    ])

    # ── Selección final de columnas ──────────────────────────────────────
    ID_COLS = ['race_name', 'session_key', 'driver_number', 'lap_number']

    FEATURE_COLS = [
        # Ritmo
        'lap_duration', 'duration_sector_1', 'duration_sector_2', 'duration_sector_3',
        # Velocidad
        'st_speed', 'i1_speed', 'i2_speed',
        # Estilo
        'throttle_mean_lap', 'throttle_pct_full', 'brake_max_lap', 'coasting_pct',
        # Motor
        'rpm_max_lap', 'n_gear_max_lap',
        # Neumático
        'tyre_age', 'compound_ord', 'lap_vs_best_stint',
        # Posición
        'position', 'gap_to_leader',
        # Stint
        'stint_number', 'is_pit_out_lap', 'is_pit_lap',
        # Eficiencias derivadas
        'speed_per_rpm', 'sector1_pct', 'sector3_pct', 'sector_balance',
        'throttle_brake_ratio', 'best_lap_stint',
    ]

    # Sólo columnas que existen en el df
    available = ID_COLS + [c for c in FEATURE_COLS if c in df.columns]
    return df.select(available)


df_telem_feat = build_telemetry_features(df_master)

print(f'Features de telemetría: {df_telem_feat.shape}')
print(f'Columnas: {[c for c in df_telem_feat.columns if c not in ["race_name","driver_number","lap_number"]]}')

Features de telemetría: (3331, 30)
Columnas: ['session_key', 'lap_duration', 'duration_sector_1', 'duration_sector_2', 'duration_sector_3', 'st_speed', 'i1_speed', 'i2_speed', 'throttle_mean_lap', 'throttle_pct_full', 'brake_max_lap', 'coasting_pct', 'rpm_max_lap', 'n_gear_max_lap', 'tyre_age', 'compound_ord', 'lap_vs_best_stint', 'position', 'stint_number', 'is_pit_out_lap', 'is_pit_lap', 'speed_per_rpm', 'sector1_pct', 'sector3_pct', 'sector_balance', 'throttle_brake_ratio', 'best_lap_stint']


In [36]:
# ── Diagnóstico de nulos (telemetría) ──────────────────────────────────────
import pandas as pd

df_telem_pd = df_telem_feat.to_pandas()
null_pct = df_telem_pd.isnull().mean().sort_values(ascending=False)
print('Columnas con > 30% nulos:')
print(null_pct[null_pct > 0.3])
print(f'\nFeatures limpias (< 30% nulos): {(null_pct <= 0.3).sum()} / {len(null_pct)}')

Columnas con > 30% nulos:
speed_per_rpm           0.374662
brake_max_lap           0.371060
throttle_mean_lap       0.371060
throttle_brake_ratio    0.371060
rpm_max_lap             0.371060
throttle_pct_full       0.371060
n_gear_max_lap          0.371060
dtype: float64

Features limpias (< 30% nulos): 23 / 30


In [38]:
# ── Exportar features de telemetría ──────────────────────────────────────
out_telem = '../../data/features/telemetry_features_v4.parquet'
#out_telem.parent.mkdir(parents=True, exist_ok=True)
df_telem_feat.write_parquet(str(out_telem))
print(f'Guardado: {out_telem}')
print(f'Shape   : {df_telem_feat.shape}')

Guardado: ../../data/features/telemetry_features_v4.parquet
Shape   : (3331, 30)


In [39]:
df_telem_feat.columns

['race_name',
 'session_key',
 'driver_number',
 'lap_number',
 'lap_duration',
 'duration_sector_1',
 'duration_sector_2',
 'duration_sector_3',
 'st_speed',
 'i1_speed',
 'i2_speed',
 'throttle_mean_lap',
 'throttle_pct_full',
 'brake_max_lap',
 'coasting_pct',
 'rpm_max_lap',
 'n_gear_max_lap',
 'tyre_age',
 'compound_ord',
 'lap_vs_best_stint',
 'position',
 'stint_number',
 'is_pit_out_lap',
 'is_pit_lap',
 'speed_per_rpm',
 'sector1_pct',
 'sector3_pct',
 'sector_balance',
 'throttle_brake_ratio',
 'best_lap_stint']

---
## 3. CAPA B — Features tácticas (para Clustering)

**Unidad:** 1 fila = 1 evento táctico (overtake / undercut / overcut)  
**Objetivo:** ~25 variables que describan el *contexto estratégico* del evento  
**Incluye:** diferenciales atacante vs defensor, contexto de stint, fase de carrera

### Variables objetivo (25 total)

| Grupo | Variables | N |
|---|---|---|
| Diferencial de ritmo (3v) | `delta_lap_mean`, `delta_sector1_mean`, `delta_sector3_mean` | 3 |
| Tendencia de ritmo | `att_lap_slope`, `def_lap_slope`, `delta_lap_slope` | 3 |
| Neumático | `att_tyre_age`, `def_tyre_age`, `delta_tyre_age`, `att_compound_ord`, `def_compound_ord`, `delta_compound_ord` | 6 |
| Degradación | `att_deg_rate`, `def_deg_rate`, `delta_deg_rate` | 3 |
| Contexto de carrera | `race_pct_complete`, `laps_remaining`, `position_gap`, `is_top10_battle` | 4 |
| Pit strategy | `att_is_pit_out`, `def_is_pit_out`, `att_stint_laps_done`, `def_stint_laps_done` | 4 |
| Identificadores (no entran al clustering) | `event_id`, `race_name`, `event_type`, `pos_change` | — |

In [40]:
# ── Helpers ────────────────────────────────────────────────────────────────

def linreg_slope(values: list) -> float | None:
    """Pendiente de regresión lineal simple. Retorna None si < 2 valores."""
    vals = [v for v in values if v is not None]
    if len(vals) < 2:
        return None
    x = np.arange(len(vals), dtype=float)
    y = np.array(vals, dtype=float)
    return float(np.polyfit(x, y, 1)[0])


def get_window(df_race: pl.DataFrame, driver, lap: int, window: int) -> pl.DataFrame:
    """Últimas `window` vueltas del piloto antes del evento."""
    target = list(range(int(lap) - window, int(lap)))
    return df_race.filter(
        (pl.col('driver_number') == driver) &
        (pl.col('lap_number').is_in(target))
    )


def col_mean(df: pl.DataFrame, col: str) -> float | None:
    if col not in df.columns or df.is_empty():
        return None
    v = df.select(pl.col(col).mean()).item()
    return float(v) if v is not None else None


def col_item(df: pl.DataFrame, col: str) -> float | None:
    if col not in df.columns or df.is_empty():
        return None
    try:
        v = df.select(pl.col(col).first()).item()
        return float(v) if v is not None else None
    except Exception:
        return None


def slope_col(df: pl.DataFrame, col: str) -> float | None:
    if col not in df.columns or df.is_empty():
        return None
    return linreg_slope(df[col].to_list())


def delta(a, b):
    if a is None or b is None:
        return None
    return a - b


COMPOUND_ORD = {'SOFT': 1, 'MEDIUM': 2, 'HARD': 3, 'INTERMEDIATE': 1.5, 'WET': 0.5}
print('Helpers cargados OK')

Helpers cargados OK


In [41]:
# ── Función principal de features tácticas ─────────────────────────────────

def extract_tactical_features(event: dict, df_race: pl.DataFrame,
                               df_events_race: pl.DataFrame,
                               total_laps: int) -> dict:
    """
    Extrae ~25 features tácticas para un evento dado.
    """
    lap = int(event['lap_number'])
    att = event['initiator_driver']
    dfn = event['target_driver']
    race = event['race_name']

    feat = {
        # ── Identificadores (no entran al modelo) ──────────────────────────
        'event_id'  : f"{race}_L{lap}_{att}v{dfn}",
        'race_name' : race,
        'event_type': event.get('event_type'),
        'pos_change': event.get('initiator_pos_change'),
        'attacker'  : att,
        'defender'  : dfn,
        'lap_number': lap,
    }

    # Ventanas de 3 vueltas
    att3 = get_window(df_race, att, lap, 3)
    def3 = get_window(df_race, dfn, lap, 3)

    # Vuelta inmediatamente anterior
    att_prev = df_race.filter(
        (pl.col('driver_number') == att) & (pl.col('lap_number') == lap - 1)
    )
    def_prev = df_race.filter(
        (pl.col('driver_number') == dfn) & (pl.col('lap_number') == lap - 1)
    )

    # ── GRUPO 1: Diferencial de ritmo (3 vueltas) ──────────────────────────
    att_lap = col_mean(att3, 'lap_duration')
    def_lap = col_mean(def3, 'lap_duration')
    feat['att_lap_mean']   = att_lap
    feat['def_lap_mean']   = def_lap
    feat['delta_lap_mean'] = delta(att_lap, def_lap)

    att_s1 = col_mean(att3, 'duration_sector_1')
    def_s1 = col_mean(def3, 'duration_sector_1')
    feat['delta_sector1_mean'] = delta(att_s1, def_s1)

    att_s3 = col_mean(att3, 'duration_sector_3')
    def_s3 = col_mean(def3, 'duration_sector_3')
    feat['delta_sector3_mean'] = delta(att_s3, def_s3)

    # ── GRUPO 2: Tendencia de ritmo (slope en 3 vueltas) ──────────────────
    att_slope = slope_col(att3, 'lap_duration')
    def_slope = slope_col(def3, 'lap_duration')
    feat['att_lap_slope']   = att_slope
    feat['def_lap_slope']   = def_slope
    feat['delta_lap_slope'] = delta(att_slope, def_slope)
    # slope negativo = mejorando (lap_duration bajando) → atacante acelerando

    # ── GRUPO 3: Neumático ─────────────────────────────────────────────────
    att_age  = col_item(att_prev, 'tyre_age')
    def_age  = col_item(def_prev, 'tyre_age')
    att_comp = COMPOUND_ORD.get(col_item(att_prev, 'compound'))
    def_comp = COMPOUND_ORD.get(col_item(def_prev, 'compound'))

    feat['att_tyre_age']      = att_age
    feat['def_tyre_age']      = def_age
    feat['delta_tyre_age']    = delta(att_age, def_age)
    feat['att_compound_ord']  = att_comp
    feat['def_compound_ord']  = def_comp
    feat['delta_compound_ord'] = delta(att_comp, def_comp)

    # ── GRUPO 4: Degradación en el stint actual ────────────────────────────
    for prefix, driver in [('att', att), ('def', dfn)]:
        stint_id = col_item(
            df_race.filter((pl.col('driver_number') == driver) & (pl.col('lap_number') == lap - 1)),
            'stint_number'
        )
        if stint_id is not None and 'stint_number' in df_race.columns:
            stint_data = df_race.filter(
                (pl.col('driver_number') == driver) &
                (pl.col('stint_number') == int(stint_id)) &
                (pl.col('lap_number') < lap)
            )
            vals = [v for v in stint_data['lap_duration'].to_list() if v is not None] \
                   if 'lap_duration' in stint_data.columns else []
            feat[f'{prefix}_deg_rate']        = linreg_slope(vals)   # s/vuelta de pérdida de ritmo
            feat[f'{prefix}_stint_laps_done'] = len(vals)
        else:
            feat[f'{prefix}_deg_rate']        = None
            feat[f'{prefix}_stint_laps_done'] = None

    feat['delta_deg_rate'] = delta(feat.get('att_deg_rate'), feat.get('def_deg_rate'))

    # ── GRUPO 5: Contexto de carrera ───────────────────────────────────────
    race_pct = lap / total_laps if total_laps else None
    feat['race_pct_complete'] = race_pct
    feat['laps_remaining']    = total_laps - lap if total_laps else None

    att_pos = col_item(att_prev, 'position')
    def_pos = col_item(def_prev, 'position')
    feat['position_gap']    = delta(def_pos, att_pos)   # positivo = defensor delante
    feat['is_top10_battle'] = (
        int(att_pos <= 10 and def_pos <= 10)
        if att_pos is not None and def_pos is not None else None
    )

    # ── GRUPO 6: Pit strategy ──────────────────────────────────────────────
    feat['att_is_pit_out'] = (
        int(col_item(att_prev, 'is_pit_out_lap') == 1.0)
        if 'is_pit_out_lap' in att_prev.columns and not att_prev.is_empty() else 0
    )
    feat['def_is_pit_out'] = (
        int(col_item(def_prev, 'is_pit_out_lap') == 1.0)
        if 'is_pit_out_lap' in def_prev.columns and not def_prev.is_empty() else 0
    )

    return feat


print('Función de features tácticas lista')

Función de features tácticas lista


In [42]:
# ── Ejecución del feature engineering táctico ──────────────────────────────
print('Extrayendo features tácticas...')

features_list = []

for race in CARRERAS:
    df_race   = df_master.filter(pl.col('race_name') == race)
    df_ev_r   = df_events.filter(pl.col('race_name') == race)
    total_laps = df_race.select(pl.col('lap_number').max()).item()

    events_race = df_ev_r.to_dicts()
    print(f'  {race}: {len(events_race)} eventos, {total_laps} vueltas')

    for event in events_race:
        feat = extract_tactical_features(event, df_race, df_ev_r, total_laps)
        features_list.append(feat)

import pandas as pd
df_tactical_feat = pd.DataFrame(features_list)

print(f'\n✅ Completado')
print(f'   Eventos    : {len(df_tactical_feat)}')
print(f'   Variables  : {len(df_tactical_feat.columns)}')

Extrayendo features tácticas...
  australia: 120 eventos, 57.0 vueltas
  china: 61 eventos, 56.0 vueltas
  japan: 53 eventos, 53.0 vueltas
  united_states: 409 eventos, 57.0 vueltas

✅ Completado
   Eventos    : 643
   Variables  : 32


In [43]:
# ── Diagnóstico de nulos (tácticas) ───────────────────────────────────────
ID_COLS = ['event_id', 'race_name', 'event_type', 'pos_change', 'attacker', 'defender', 'lap_number']
feat_cols = [c for c in df_tactical_feat.columns if c not in ID_COLS]

null_pct = df_tactical_feat[feat_cols].isnull().mean().sort_values(ascending=False)
print('Nulos por feature:')
print(null_pct.to_string())
print(f'\nFeatures limpias (< 30% nulos): {(null_pct <= 0.3).sum()} / {len(null_pct)}')

# Resumen por grupo
grupos = {
    'Ritmo (3v)' : ['att_lap_mean', 'def_lap_mean', 'delta_lap_mean', 'delta_sector1_mean', 'delta_sector3_mean'],
    'Tendencia'  : ['att_lap_slope', 'def_lap_slope', 'delta_lap_slope'],
    'Neumático'  : ['att_tyre_age', 'def_tyre_age', 'delta_tyre_age', 'att_compound_ord', 'def_compound_ord', 'delta_compound_ord'],
    'Degradación': ['att_deg_rate', 'def_deg_rate', 'delta_deg_rate', 'att_stint_laps_done', 'def_stint_laps_done'],
    'Contexto'   : ['race_pct_complete', 'laps_remaining', 'position_gap', 'is_top10_battle'],
    'Pit'        : ['att_is_pit_out', 'def_is_pit_out'],
}
print('\nResumen por grupo:')
total = 0
for g, cols in grupos.items():
    n = len([c for c in cols if c in df_tactical_feat.columns])
    print(f'  {g:15s}: {n} features')
    total += n
print(f'  TOTAL: {total} features tácticas')

Nulos por feature:
def_compound_ord       1.000000
att_compound_ord       1.000000
delta_compound_ord     1.000000
delta_deg_rate         0.146190
def_deg_rate           0.135303
def_lap_slope          0.129082
delta_lap_slope        0.129082
def_tyre_age           0.122862
def_stint_laps_done    0.122862
is_top10_battle        0.122862
position_gap           0.122862
delta_tyre_age         0.122862
delta_sector3_mean     0.091757
delta_lap_mean         0.091757
def_lap_mean           0.091757
delta_sector1_mean     0.091757
att_deg_rate           0.051322
att_lap_slope          0.037325
att_tyre_age           0.031104
att_stint_laps_done    0.031104
att_lap_mean           0.000000
laps_remaining         0.000000
race_pct_complete      0.000000
att_is_pit_out         0.000000
def_is_pit_out         0.000000

Features limpias (< 30% nulos): 22 / 25

Resumen por grupo:
  Ritmo (3v)     : 5 features
  Tendencia      : 3 features
  Neumático      : 6 features
  Degradación    : 5 features


In [44]:
# ── Exportar features tácticas ─────────────────────────────────────────────
out_tact = BASE_DIR /'features' / 'tactical_features_v4.parquet'
out_tact.parent.mkdir(parents=True, exist_ok=True)

pl.from_pandas(df_tactical_feat).write_parquet(str(out_tact))
print(f'Guardado: {out_tact}')
print(f'Shape   : {df_tactical_feat.shape}')

df_tactical_feat.head(3)

Guardado: ..\..\data\features\tactical_features_v4.parquet
Shape   : (643, 32)


,event_id,race_name,event_type,pos_change,attacker,defender,lap_number,att_lap_mean,def_lap_mean,delta_lap_mean,...,att_stint_laps_done,def_deg_rate,def_stint_laps_done,delta_deg_rate,race_pct_complete,laps_remaining,position_gap,is_top10_battle,att_is_pit_out,def_is_pit_out
0,australia_L4_1.0v10.0,australia,On_Track_Overtake,P10 -> P7,1.0,10.0,4,86.36,86.1325,0.2275,...,2.0,-0.477,2.0,-0.529,0.070175,53.0,-3.0,1.0,0,0
1,australia_L4_1.0v87.0,australia,On_Track_Overtake,P10 -> P7,1.0,87.0,4,86.36,86.1520,0.2080,...,2.0,-0.464,2.0,-0.542,0.070175,53.0,-2.0,1.0,0,0
2,australia_L4_1.0v5.0,australia,On_Track_Overtake,P10 -> P7,1.0,5.0,4,86.36,86.1780,0.1820,...,2.0,-0.292,2.0,-0.714,0.070175,53.0,-1.0,1.0,0,0


---
## 4. Verificación final

Chequeo rápido de que ambos datasets tienen la proporción correcta de features vs filas.

In [45]:
# ── Resumen final ──────────────────────────────────────────────────────────
telem_pd = df_telem_feat.to_pandas()
telem_feat_cols = [c for c in telem_pd.columns if c not in ['race_name', 'driver_number', 'lap_number']]
tact_feat_cols  = [c for c in df_tactical_feat.columns if c not in ID_COLS]

print('=' * 55)
print('RESUMEN FINAL DE FEATURE ENGINEERING v4')
print('=' * 55)
print(f'\nCAPA A — Telemetría (input para PCA):')
print(f'  Filas     : {len(telem_pd)}')
print(f'  Features  : {len(telem_feat_cols)}')
print(f'  Ratio     : {len(telem_pd) // len(telem_feat_cols)}:1 (recomendado > 50:1)')

print(f'\nCAPA B — Táctica (input para Clustering):')
print(f'  Filas     : {len(df_tactical_feat)}')
print(f'  Features  : {len(tact_feat_cols)}')
print(f'  Ratio     : {len(df_tactical_feat) // len(tact_feat_cols)}:1 (recomendado > 20:1)')

print(f'\nArchivos generados:')
print(f'  {out_telem}')
print(f'  {out_tact}')
print('\nSiguiente paso → PCA sobre telemetry_features_v4.parquet')

RESUMEN FINAL DE FEATURE ENGINEERING v4

CAPA A — Telemetría (input para PCA):
  Filas     : 3331
  Features  : 27
  Ratio     : 123:1 (recomendado > 50:1)

CAPA B — Táctica (input para Clustering):
  Filas     : 643
  Features  : 25
  Ratio     : 25:1 (recomendado > 20:1)

Archivos generados:
  ../../data/features/telemetry_features_v4.parquet
  ..\..\data\features\tactical_features_v4.parquet

Siguiente paso → PCA sobre telemetry_features_v4.parquet
